# TinyMWAMNet · Notebook 02 — Quantising the MWAMNet teacher (TFLite ladder)

The teacher (DenseNet201 + MobileNetV3-Large, 4.5 GMAC) is converted to **FP16**, **dynamic-range INT8** and **full-integer INT8** with two calibration sets (in-domain vs confounder-aware images). Each deployed `.tflite` is evaluated on the full in-domain test set and on fixed, label-stratified random subsets of 1,000 images from each shifted dataset (`LADDER_N`), because full-integer inference of a model this size runs at only ~1–4 images/s on Colab's CPU. INT8 variants use 3 seeds including the median seed; FP16 and dynamic-range use the median seed. It finally measures CPU latency, the TensorFlow-Lite-Micro tensor arena and the Arm Ethos-U55 (Vela) estimate.


In [ ]:
#@title 1 · Install packages
import importlib.metadata as md, subprocess, sys, os
def _v(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None
tfv = _v("tensorflow")
pk = []
if tfv is None:                         # Colab normally ships TensorFlow; install if absent
    pk.append("tensorflow[and-cuda]==2.21.0"); tfv = "2.21.0"
mm = ".".join(tfv.split(".")[:2])
pk += [f"tf_keras~={mm}.0", "setuptools<81"] + ['tflite-micro==0.dev20251201172741', 'ethos-u-vela==5.2.0', 'statsmodels', 'ai-edge-litert']
if not os.environ.get("TMN_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pk, check=True)
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # tf_keras (Keras 2) is required by TF-MOT for QAT
print("TensorFlow", tfv, "| installed:", pk)

TensorFlow 2.20.0 | installed: ['tf_keras~=2.20.0', 'setuptools<81', 'tflite-micro==0.dev20251201172741', 'ethos-u-vela==5.2.0', 'statsmodels', 'ai-edge-litert']


In [ ]:
%%writefile /content/tmn_common.py
"""
TinyMWAMNet — shared utilities for all Colab notebooks.

"""
import os, sys, json, math, time, glob, hashlib, random, gc, re, subprocess, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np

VERSION = "29 Sep 2026"      # printed by cell 3 of every notebook

# ---------------------------------------------------------------------------
# 1. Configuration (pre-specified; nothing here is tuned on shifted data)
# ---------------------------------------------------------------------------
CFG = dict(
    # Kaggle dataset identifiers
    DIN_ID="alik05/forest-fire-dataset",                 # in-domain (paper 1)
    WILD_ID="elmadafri/the-wildfire-dataset",            # Shift-1 (anticipated)
    DFIRE_ID="sayedgamal99/smoke-fire-detection-yolo",   # Shift-2 (unanticipated, CC0)
    IMG=224, SEED=42, SEEDS=[42, 43, 44, 45, 46], VAL_FRAC=0.20,
    # teacher (identical to paper 1)
    TEACHER_COPIES=4, HUE=0.10, L2=0.01, DROPOUT=0.5, LR=1e-4, WD=1e-4,
    EPOCHS=100, BATCH=32, PATIENCE=15,
    # augmentation pools for students
    STD_COPIES=8,          # paper-1 augmentation; copies 1-4 are identical to paper 1
    CONF_COPIES=4,         # confounder-aware copies (CAD)
    # students
    KD_T=2.0, KD_ALPHA=0.5,
    ST_BATCH=64, ST_WARM_EPOCHS=3, ST_EPOCHS=25, ST_LR_WARM=1e-3, ST_LR=3e-4,
    ST_WD=1e-4, ST_DROPOUT=0.2, QAT_EPOCHS=5, QAT_LR=5e-5,
    CALIB_N=256,
    # leakage audit
    PHASH_MAX_DIST=6,
    # weight-only bit-width sweep
    WBITS=[8, 6, 5, 4, 3],
    TEST_MODE=bool(int(os.environ.get("TMN_TEST_MODE", "0"))),
)

MAIN_STUDENT = ("mnv3s", 1.0, 128)          # TinyMWAMNet-S
FAMILY = [("mnv1", 0.25, 96),                # TinyMWAMNet-XS (MLPerf Tiny VWW size)
          ("mnv3s", 1.0, 96), ("mnv3s", 1.0, 160), ("mnv2", 0.35, 128)]
REGIMES = {  # name: (use KD?, pool kind)
    "CE-std": (False, "std"), "CE-conf": (False, "conf"),
    "KD-std": (True, "std"), "CAD": (True, "conf"),   # CAD = KD + confounder aug
}
EVALSETS = ("din", "wild", "dfire")
EVAL_NAMES = {"din": "In-domain test", "wild": "Shift-1 (The Wildfire Dataset)",
              "dfire": "Shift-2 (D-Fire)"}

if CFG["TEST_MODE"]:
    CFG.update(SEEDS=[42, 43], EPOCHS=3, PATIENCE=2, STD_COPIES=2, CONF_COPIES=2,
               TEACHER_COPIES=1, ST_WARM_EPOCHS=1, ST_EPOCHS=1, QAT_EPOCHS=1,
               CALIB_N=8, WBITS=[8, 4])
    FAMILY = [("mnv1", 0.25, 96)]

IMGEXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


# ---------------------------------------------------------------------------
# 2. Paths (Google Drive if mounted, else local)
# ---------------------------------------------------------------------------
def setup_paths():
    root = os.environ.get("TMN_ROOT")
    if not root:
        root = ("/content/drive/MyDrive/TinyMWAMNet"
                if os.path.isdir("/content/drive/MyDrive") else "/content/TinyMWAMNet")
    P = {k: os.path.join(root, k) for k in
         ["data", "features", "models", "results", "preds", "figures", "tflite", "logs"]}
    P["root"] = root
    P["tmp"] = os.environ.get("TMN_TMP", "/content/tmn_tmp")
    for v in P.values():
        os.makedirs(v, exist_ok=True)
    return P


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))


def load_json(path):
    with open(path) as f:
        return json.load(f)


def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    try:
        import tf_keras as keras
        keras.utils.set_random_seed(seed)
    except Exception:
        pass


# ---------------------------------------------------------------------------
# 3. Dataset download and discovery
# ---------------------------------------------------------------------------
def kaggle_download(kid):
    """kagglehub download; TMN_FAKE_DATA=<dir> redirects to local copies (testing)."""
    fake = os.environ.get("TMN_FAKE_DATA")
    if fake:
        return os.path.join(fake, kid.replace("/", "__"))
    import kagglehub
    return kagglehub.dataset_download(kid)


def _load_one(p, S, draft):
    from PIL import Image
    try:
        with Image.open(p) as im:
            if draft:
                im.draft("RGB", (S * 2, S * 2))       # fast DCT-scaled JPEG decode
            return np.asarray(im.convert("RGB").resize((S, S), Image.BILINEAR))
    except Exception as e:
        print("   unreadable, stored as black frame:", os.path.basename(p), e)
        return np.zeros((S, S, 3), np.uint8)


def load_images(paths, S=224, draft=False, workers=8):
    out = np.zeros((len(paths), S, S, 3), np.uint8)
    with ThreadPoolExecutor(workers) as ex:
        for i, a in enumerate(ex.map(lambda p: _load_one(p, S, draft), paths)):
            out[i] = a
    return out


def discover_din(root):
    """Forest Fire Dataset (alik05). Identical file lists and split to paper 1."""
    from sklearn.model_selection import train_test_split
    base = glob.glob(os.path.join(root, "**", "Forest Fire Dataset"), recursive=True)[0]
    tr_dir, te_dir = os.path.join(base, "Training"), os.path.join(base, "Testing")
    pool_p, pool_y = [], []
    for cls, lab in (("nofire", 0), ("fire", 1)):
        fs = sorted(glob.glob(os.path.join(tr_dir, cls, "*.jpg")))
        pool_p += fs; pool_y += [lab] * len(fs)
    pool_y = np.array(pool_y, np.int32)
    te_p = sorted(glob.glob(os.path.join(te_dir, "*.jpg")))
    te_y = np.array([0 if os.path.basename(p).lower().startswith("nofire") else 1
                     for p in te_p], np.int32)
    tr_p, va_p, tr_y, va_y = train_test_split(pool_p, pool_y, test_size=CFG["VAL_FRAC"],
                                              random_state=CFG["SEED"], stratify=pool_y)
    return dict(tr_p=list(tr_p), va_p=list(va_p), te_p=te_p,
                tr_y=np.array(tr_y, np.int32), va_y=np.array(va_y, np.int32), te_y=te_y)


def discover_wild(root):
    """The Wildfire Dataset: labels from 'fire'/'nofire' folders, subclass = leaf folder."""
    recs = []
    for dp, dn, fn in os.walk(root):
        dn.sort()
        parts = [p.lower() for p in dp[len(root):].split(os.sep) if p]
        if any(p == "nofire" for p in parts):
            lab = 0
        elif any(p == "fire" for p in parts):
            lab = 1
        else:
            continue
        sub = parts[-1] if parts[-1] not in ("fire", "nofire") else "unspecified"
        for f in sorted(fn):
            if os.path.splitext(f)[1].lower() in IMGEXT:
                recs.append((os.path.join(dp, f), lab, sub))
    recs.sort(key=lambda r: r[0])
    return dict(paths=[r[0] for r in recs], y=np.array([r[1] for r in recs], np.int32),
                sub=np.array([r[2] for r in recs]))


def _yolo_names(root):
    try:
        import yaml
    except Exception:
        return None
    for y in glob.glob(os.path.join(root, "**", "*.yaml"), recursive=True):
        try:
            d = yaml.safe_load(open(y))
            names = d.get("names") if isinstance(d, dict) else None
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names)]
            if names:
                return [str(n).lower() for n in names]
        except Exception:
            pass
    return None


def discover_dfire(root, prefer=("test",)):
    """D-Fire (YOLO format). fire = >=1 fire box; no-fire = no boxes; smoke-only excluded."""
    lab_dirs = [d for d in glob.glob(os.path.join(root, "**", "labels"), recursive=True)
                if os.path.isdir(os.path.join(os.path.dirname(d), "images"))]
    splits = {os.path.basename(os.path.dirname(d)).lower(): d for d in lab_dirs}
    print("D-Fire splits found:", {k: len(os.listdir(v)) for k, v in splits.items()})
    chosen = [splits[k] for k in prefer if k in splits] or list(splits.values())
    names = _yolo_names(root)
    if names and "fire" in names:
        fire_id = names.index("fire")
    else:
        fire_id = 1                                   # D-Fire convention: 0 smoke, 1 fire
    recs, stats = [], dict(fire=0, none=0, smoke_only=0)
    only = {}
    for ld in chosen:
        imd = os.path.join(os.path.dirname(ld), "images")
        imgs = {os.path.splitext(f)[0]: os.path.join(imd, f) for f in os.listdir(imd)
                if os.path.splitext(f)[1].lower() in IMGEXT}
        for stem, ip in sorted(imgs.items()):
            lf = os.path.join(ld, stem + ".txt")
            cls = []
            if os.path.exists(lf):
                cls = [int(float(l.split()[0])) for l in open(lf) if l.strip()]
            s = set(cls)
            if len(s) == 1:
                only[next(iter(s))] = only.get(next(iter(s)), 0) + 1
            if fire_id in s:
                recs.append((ip, 1, "fire+smoke" if len(s) > 1 else "fire_only")); stats["fire"] += 1
            elif not s:
                recs.append((ip, 0, "none")); stats["none"] += 1
            else:
                stats["smoke_only"] += 1
    print("D-Fire class names:", names, "-> fire id =", fire_id,
          "| images with a single class:", only, "| kept:", stats)
    if only and len(only) == 2 and only.get(fire_id, 0) > only.get(1 - fire_id, 0):
        print("WARNING: the class treated as 'fire' is the more frequent single class; in "
              "D-Fire smoke-only images outnumber fire-only images ~5:1. Check the mapping!")
    return dict(paths=[r[0] for r in recs], y=np.array([r[1] for r in recs], np.int32),
                sub=np.array([r[2] for r in recs]), stats=stats, fire_id=fire_id)


# ---------------------------------------------------------------------------
# 4. Augmentation (paper-1 recipe) and confounder-aware augmentation (CAD)
# ---------------------------------------------------------------------------
def _tf():
    import tensorflow as tf
    return tf


def std_aug_block(X, copy_idx, indices=None, batch=256):
    """Paper-1 augmentation, seeds s=[i, c*7919+13]; returns uint8 array.
    Copies 1..4 are bit-identical to the paper-1 teacher training pool."""
    tf = _tf()
    idx = np.arange(len(X)) if indices is None else np.asarray(indices)
    c = int(copy_idx)

    def aug_one(x_u8, i):
        s = tf.stack([tf.cast(i, tf.int32), tf.constant(c * 7919 + 13, tf.int32)])
        im = tf.cast(x_u8, tf.float32) / 255.0
        im = tf.image.stateless_random_flip_left_right(im, s)
        im = tf.image.stateless_random_brightness(im, 0.10, s + 1)
        im = tf.image.stateless_random_contrast(im, 0.9, 1.1, s + 2)
        im = tf.image.stateless_random_saturation(im, 0.9, 1.1, s + 3)
        im = tf.image.stateless_random_hue(im, CFG["HUE"], s + 4)
        im = tf.clip_by_value(im, 0.0, 1.0)
        im = tf.image.stateless_random_jpeg_quality(im, 75, 100, s + 5)
        return tf.cast(tf.round(im * 255.0), tf.uint8)

    ds = tf.data.Dataset.from_tensor_slices((X[idx], idx.astype(np.int32)))
    ds = ds.map(aug_one, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True).batch(batch)
    with tf.device("/CPU:0"):
        return np.concatenate([b.numpy() for b in ds]) if len(idx) else X[:0].copy()


# ---- confounder transforms (NumPy/SciPy only, deterministic per image) ----
def _warm_cast(im, r):
    """Sunset / golden-hour illumination: channel gains plus a warm sky gradient."""
    g = np.array([r.uniform(1.05, 1.35), r.uniform(0.92, 1.08), r.uniform(0.60, 0.88)])
    out = im * g
    H = im.shape[0]
    ramp = np.linspace(1.0, 0.0, H)[:, None, None] ** r.uniform(0.8, 2.0)
    a = r.uniform(0.15, 0.45) * ramp
    tint = np.array([1.0, r.uniform(0.45, 0.65), r.uniform(0.10, 0.30)])
    return out * (1 - a) + tint * a


def _foliage(im, r):
    """Autumn foliage: rotate green hues towards yellow/red, boost saturation.
    Pure NumPy (matplotlib's HSV conversion); no OpenCV dependency."""
    from matplotlib.colors import rgb_to_hsv, hsv_to_rgb
    hsv = rgb_to_hsv(np.clip(im, 0.0, 1.0).astype(np.float64))
    h = hsv[..., 0] * 360.0                                         # hue in degrees
    green = (h > 55) & (h < 170)
    h[green] = (h[green] - r.uniform(40, 95)) % 360.0
    hsv[..., 0] = h / 360.0
    hsv[..., 1] = np.clip(hsv[..., 1] * np.where(green, r.uniform(1.0, 1.3), 1.0), 0, 1)
    return hsv_to_rgb(hsv)


def _haze(im, r):
    """Fog / haze via the atmospheric scattering model I = J t + A (1 - t)."""
    H = im.shape[0]
    t0 = r.uniform(0.45, 0.85)
    grad = np.linspace(r.uniform(0.0, 0.25), 0.0, H)[:, None, None]  # denser near the top
    t = np.clip(t0 - grad, 0.25, 1.0)
    A = r.uniform(0.75, 0.95) * np.array([1.0, 1.0, r.uniform(0.95, 1.05)])
    return im * t + A * (1 - t)


def _glare(im, r):
    """Sun glare / specular reflection: additive warm Gaussian blob."""
    H, W = im.shape[:2]
    cy, cx = r.uniform(0.05, 0.95) * H, r.uniform(0.05, 0.95) * W
    s = r.uniform(0.08, 0.25) * H
    yy, xx = np.mgrid[0:H, 0:W]
    g = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s * s))[..., None]
    col = np.array([1.0, r.uniform(0.80, 0.95), r.uniform(0.50, 0.75)])
    return im + r.uniform(0.3, 0.8) * g * col


def _line_kernel(L, angle_deg):
    """Normalised L x L linear motion-blur kernel at the given angle (bilinear splatting)."""
    k = np.zeros((L, L), np.float64)
    c = (L - 1) / 2.0
    t = np.linspace(-c, c, 8 * L)
    a = np.deg2rad(angle_deg)
    xs, ys = c + t * np.cos(a), c - t * np.sin(a)
    x0, y0 = np.floor(xs).astype(int), np.floor(ys).astype(int)
    fx, fy = xs - x0, ys - y0
    for dx, dy, w in ((0, 0, (1 - fx) * (1 - fy)), (1, 0, fx * (1 - fy)),
                      (0, 1, (1 - fx) * fy), (1, 1, fx * fy)):
        np.add.at(k, (np.clip(y0 + dy, 0, L - 1), np.clip(x0 + dx, 0, L - 1)), w)
    return k / k.sum()


def _motion(im, r):
    """Camera/platform motion blur: linear kernel of random length and angle.
    SciPy convolution with reflected borders; no OpenCV dependency."""
    from scipy.ndimage import convolve
    L = int(r.choice([5, 7, 9, 11, 13, 15]))
    k = _line_kernel(L, r.uniform(0, 180))
    return np.stack([convolve(im[..., ch].astype(np.float64), k, mode="reflect")
                     for ch in range(im.shape[-1])], axis=-1)


def _lowlight(im, r):
    """Dusk / low light: gamma darkening, slight blue shift and sensor noise."""
    out = im ** r.uniform(1.6, 2.6)
    out = out * np.array([r.uniform(0.85, 1.0), r.uniform(0.9, 1.0), 1.0])
    return out + r.normal(0, r.uniform(0.01, 0.04), im.shape)


CONF_OPS = [("warm_cast", _warm_cast), ("autumn_foliage", _foliage), ("haze", _haze),
            ("glare", _glare), ("motion_blur", _motion), ("low_light", _lowlight)]


def conf_one(img_u8, i, k, force=None):
    """Label-preserving confounder transform of one image (i = index, k = copy)."""
    r = np.random.default_rng([int(i), int(k), 2026])
    im = img_u8.astype(np.float32) / 255.0
    if force is not None:
        ops = [dict(CONF_OPS)[force]]
    else:
        n = 1 if r.uniform() < 0.6 else 2
        ops = [CONF_OPS[j][1] for j in r.choice(len(CONF_OPS), n, replace=False)]
    for f in ops:
        im = np.clip(f(im, r), 0.0, 1.0)
    return np.round(im * 255.0).astype(np.uint8)


def conf_aug_block(X, copy_idx, indices=None, workers=8):
    """Confounder copy k: paper-1 augmentation (seed family 100+k) then 1-2 confounders."""
    idx = np.arange(len(X)) if indices is None else np.asarray(indices)
    base = std_aug_block(X, 100 + int(copy_idx), idx)
    out = np.empty_like(base)
    with ThreadPoolExecutor(workers) as ex:
        for j, a in enumerate(ex.map(lambda t: conf_one(t[1], t[0], copy_idx),
                                     zip(idx, base))):
            out[j] = a
    return out


def pool_blocks():
    """Ordered list of augmentation blocks forming the full student pool."""
    return ([("orig", 0)] + [("std", c) for c in range(1, CFG["STD_COPIES"] + 1)]
            + [("conf", k) for k in range(1, CFG["CONF_COPIES"] + 1)])


def regime_blocks(kind):
    """Blocks used by a student regime. Both regimes have the same number of images."""
    if kind == "std":
        return [("orig", 0)] + [("std", c) for c in range(1, CFG["STD_COPIES"] + 1)]
    half = CFG["STD_COPIES"] - CFG["CONF_COPIES"]
    return ([("orig", 0)] + [("std", c) for c in range(1, half + 1)]
            + [("conf", k) for k in range(1, CFG["CONF_COPIES"] + 1)])


def teacher_blocks():
    return [("orig", 0)] + [("std", c) for c in range(1, CFG["TEACHER_COPIES"] + 1)]


def make_block(X, kind, c, indices=None):
    if kind == "orig":
        return X if indices is None else X[np.asarray(indices)]
    if kind == "std":
        return std_aug_block(X, c, indices)
    return conf_aug_block(X, c, indices)


def array_md5(a):
    return hashlib.md5(np.ascontiguousarray(a).tobytes()).hexdigest()


def fingerprint(a, n=20000):
    """MD5 plus a fixed random sample of pixel values (to quantify tiny cross-machine drift)."""
    flat = np.ascontiguousarray(a).reshape(-1)
    idx = np.random.default_rng(0).integers(0, flat.size, n)
    return dict(md5=array_md5(flat), sample=flat[idx].astype(int).tolist())


def compare_fingerprint(a, fp):
    flat = np.ascontiguousarray(a).reshape(-1)
    if array_md5(flat) == fp["md5"]:
        return dict(identical=True, frac_diff=0.0, max_diff=0)
    idx = np.random.default_rng(0).integers(0, flat.size, len(fp["sample"]))
    d = np.abs(flat[idx].astype(int) - np.asarray(fp["sample"]))
    return dict(identical=False, frac_diff=float((d > 0).mean()), max_diff=int(d.max()))


# ---------------------------------------------------------------------------
# 5. Perceptual-hash leakage audit
# ---------------------------------------------------------------------------
def phash_array(X):
    import imagehash
    from PIL import Image
    return np.array([int(str(imagehash.phash(Image.fromarray(x))), 16) for x in X],
                    dtype=np.uint64)


def min_hamming(a, b, chunk=512):
    """For each hash in a, the minimum Hamming distance to any hash in b."""
    out = np.empty(len(a), np.int32)
    b = b.astype(np.uint64)
    for s in range(0, len(a), chunk):
        x = np.bitwise_xor(a[s:s + chunk, None].astype(np.uint64), b[None, :])
        d = np.unpackbits(x.view(np.uint8).reshape(x.shape + (8,)), axis=-1).sum(-1)
        out[s:s + chunk] = d.min(1)
    return out


# ---------------------------------------------------------------------------
# 6. Teacher (MWAMNet) — identical architecture and protocol to paper 1
# ---------------------------------------------------------------------------
def keras_mod():
    os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
    import tf_keras as keras
    return keras


def teacher_backbones(weights="imagenet"):
    keras = keras_mod()
    if CFG["TEST_MODE"]:
        weights = None
        set_seed(0)                      # deterministic random backbones in test mode
    S = CFG["IMG"]
    d = keras.applications.DenseNet201(include_top=False, weights=weights,
                                       input_shape=(S, S, 3), pooling="avg")
    m = keras.applications.MobileNetV3Large(include_top=False, weights=weights,
                                            input_shape=(S, S, 3), pooling="avg")
    d.trainable = False; m.trainable = False
    return d, m


def densenet_prep(x):
    """== keras.applications.densenet.preprocess_input (mode 'torch')."""
    tf = _tf()
    x = tf.cast(x, tf.float32) / 255.0
    return (x - tf.constant([0.485, 0.456, 0.406])) / tf.constant([0.229, 0.224, 0.225])


def mnv3_prep(x):
    """== keras.applications.mobilenet_v3.preprocess_input (identity; model rescales)."""
    return _tf().cast(x, "float32")


def extract_features(net, prep, X, bs=64):
    tf = _tf()
    out = np.empty((len(X), net.output_shape[-1]), np.float32)
    i = 0
    while i < len(X):
        try:
            b = X[i:i + bs]
            out[i:i + len(b)] = net(prep(b), training=False).numpy()
            i += len(b)
        except tf.errors.ResourceExhaustedError:
            if bs == 1:
                raise
            bs //= 2; gc.collect()
    return out


def build_head(dims=(1920, 960), l2v=None, drop=None, units=(1024, 512)):
    """Paper-1 head. Final Dense is split into 'logit' + sigmoid (mathematically identical)."""
    keras = keras_mod(); L = keras.layers
    l2v = CFG["L2"] if l2v is None else l2v
    drop = CFG["DROPOUT"] if drop is None else drop
    ins = [keras.Input(shape=(d,)) for d in dims]
    br = [L.Dropout(drop)(L.BatchNormalization()(i)) for i in ins]
    x = L.Concatenate()(br) if len(br) > 1 else br[0]
    for u in units:
        x = L.Dense(u, activation="relu", kernel_regularizer=keras.regularizers.l2(l2v))(x)
        x = L.BatchNormalization()(x)
        x = L.Dropout(drop)(x)
    z = L.Dense(1, name="logit")(x)
    p = L.Activation("sigmoid", name="prob")(z)
    return keras.Model(ins, p, name="mwamnet_head")


def head_logit_model(head):
    keras = keras_mod()
    return keras.Model(head.inputs, head.get_layer("logit").output)


def train_head(Xtr, ytr, Xva, yva, seed):
    keras = keras_mod()
    set_seed(seed)
    steps = math.ceil(len(ytr) / CFG["BATCH"]) * CFG["EPOCHS"]
    lr = keras.optimizers.schedules.CosineDecay(CFG["LR"], steps, alpha=1e-6)
    m = build_head([f.shape[1] for f in Xtr])
    m.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=CFG["WD"]),
              loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=CFG["EPOCHS"],
              batch_size=CFG["BATCH"], verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(
                  monitor="val_accuracy", mode="max", patience=CFG["PATIENCE"],
                  restore_best_weights=True)])
    return m, len(h.history["loss"])


def build_teacher_e2e(head_weights, d=None, m=None):
    """End-to-end MWAMNet: raw pixels [0,255] (224x224x3) -> P(fire)."""
    keras = keras_mod(); L = keras.layers
    if d is None:
        d, m = teacher_backbones()
    inp = keras.Input((CFG["IMG"], CFG["IMG"], 3), name="pixels")
    xd = L.Rescaling(1.0 / 255.0)(inp)
    xd = L.Normalization(mean=[0.485, 0.456, 0.406],
                         variance=[0.229 ** 2, 0.224 ** 2, 0.225 ** 2])(xd)
    fd = d(xd, training=False)
    fm = m(inp, training=False)
    head = build_head([d.output_shape[-1], m.output_shape[-1]])
    head.load_weights(head_weights)
    return keras.Model(inp, head([fd, fm]), name="MWAMNet")


# ---------------------------------------------------------------------------
# 7. Students (TinyMWAMNet)
# ---------------------------------------------------------------------------
def student_name(arch, alpha, res):
    return {"mnv3s": "MNv3S", "mnv1": "MNv1", "mnv2": "MNv2"}[arch] + f"-{alpha:g}-{res}"


def build_student(arch, alpha, res, weights="imagenet"):
    """Flat functional student core. Input in [-1, 1]; output = fire logit."""
    keras = keras_mod(); L = keras.layers
    if CFG["TEST_MODE"]:
        weights = None
    kw = dict(input_shape=(res, res, 3), include_top=False, weights=weights, pooling="avg")
    if arch == "mnv3s":
        base = keras.applications.MobileNetV3Small(alpha=alpha, minimalistic=True,
                                                   include_preprocessing=False, **kw)
    elif arch == "mnv1":
        base = keras.applications.MobileNet(alpha=alpha, **kw)
    elif arch == "mnv2":
        base = keras.applications.MobileNetV2(alpha=alpha, **kw)
    else:
        raise ValueError(arch)
    x = L.Dropout(CFG["ST_DROPOUT"], name="st_dropout")(base.output)
    z = L.Dense(1, name="logit")(x)
    core = keras.Model(base.input, z, name="core_" + student_name(arch, alpha, res).replace(".", "p"))
    return core


def set_backbone_trainable(core, flag):
    keras = keras_mod()
    for l in core.layers:
        if l.name in ("logit", "st_dropout"):
            l.trainable = True
        elif isinstance(l, keras.layers.BatchNormalization):
            l.trainable = False          # BN statistics frozen during fine-tuning
        else:
            l.trainable = flag


def kd_loss(alpha, T):
    """alpha*BCE(y, s) + (1-alpha)*T^2*BCE(sigma(t/T), sigma(s/T)); y_true=[y, t_logit]."""
    tf = _tf()

    def loss(y_true, z_s):
        y = y_true[:, 0:1]; zt = y_true[:, 1:2]
        hard = tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=z_s)
        if alpha >= 1.0:
            return tf.reduce_mean(hard)
        soft = tf.nn.sigmoid_cross_entropy_with_logits(labels=tf.sigmoid(zt / T),
                                                       logits=z_s / T) * (T * T)
        return tf.reduce_mean(alpha * hard + (1.0 - alpha) * soft)
    return loss


def to_unit(x_u8):
    """uint8 pixels -> x = (pixel - 128) / 128 in [-1, 0.992].
    With this convention the INT8 input tensor of the deployed student is exactly
    q = pixel - 128 (scale 1/128, zero point 0): one subtraction on the MCU."""
    return (x_u8.astype(np.float32) - 128.0) / 128.0


def export_wrapper(core_or_qat, res):
    """Deployed student: x = (pixel-128)/128 -> core -> sigmoid (P(fire))."""
    keras = keras_mod(); L = keras.layers
    inp = keras.Input((res, res, 3), name="pixels_centered")
    p = L.Activation("sigmoid", name="prob")(core_or_qat(inp))
    return keras.Model(inp, p)


def rep_unit(X_u8, n=None, seed=0):
    """Representative data for students (x in [-1, 0.992]); the first sample spans the full
    pixel range so that the converter derives input scale 1/128 and zero point 0 exactly."""
    n = len(X_u8) if n is None else min(n, len(X_u8))
    idx = np.random.default_rng(seed).choice(len(X_u8), n, replace=False)
    ext = np.zeros((1,) + X_u8.shape[1:], np.uint8); ext[:, ::2] = 255

    def gen():
        yield [to_unit(ext)]
        for i in idx:
            yield [to_unit(X_u8[i:i + 1])]
    return gen


def resize_u8(X, res, bs=512):
    """Antialiased bilinear downsampling of uint8 images to res x res (uint8)."""
    tf = _tf()
    if X.shape[1] == res:
        return X
    out = np.empty((len(X), res, res, 3), np.uint8)
    with tf.device("/CPU:0"):
        for s in range(0, len(X), bs):
            r = tf.image.resize(tf.cast(X[s:s + bs], tf.float32), (res, res),
                                method="bilinear", antialias=True)
            out[s:s + bs] = tf.cast(tf.clip_by_value(tf.round(r), 0, 255), tf.uint8).numpy()
    return out


# ---------------------------------------------------------------------------
# 8. TFLite conversion, LiteRT / TFLM runners, Vela
# ---------------------------------------------------------------------------
def tflite_convert(model, res, mode, rep=None, in_type="uint8"):
    """mode: fp32 | fp16 | dyn | int8 (full-integer; input uint8 (teacher, raw pixels) or
    int8 (students, pixel-128); output uint8 probability, scale 1/256)."""
    tf = _tf()

    @tf.function(input_signature=[tf.TensorSpec([1, res, res, 3], tf.float32)])
    def serve(x):
        return model(x, training=False)

    c = tf.lite.TFLiteConverter.from_concrete_functions([serve.get_concrete_function()], model)
    if mode == "fp16":
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.target_spec.supported_types = [tf.float16]
    elif mode == "dyn":
        c.optimizations = [tf.lite.Optimize.DEFAULT]
    elif mode == "int8":
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.representative_dataset = rep
        c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        c.inference_input_type = tf.int8 if in_type == "int8" else tf.uint8
        c.inference_output_type = tf.uint8
    elif mode != "fp32":
        raise ValueError(mode)
    return c.convert()


def rep_from(X_u8, n=None, seed=0):
    """Representative dataset generator (float pixels in [0,255], batch 1)."""
    n = len(X_u8) if n is None else min(n, len(X_u8))
    idx = np.random.default_rng(seed).choice(len(X_u8), n, replace=False)

    def gen():
        for i in idx:
            yield [X_u8[i:i + 1].astype(np.float32)]
    return gen


LAST_BACKEND = None     # runtime used by the most recent LiteRT call (stored with every result)


def _litert():
    try:
        from ai_edge_litert import interpreter as lit
        return lit
    except Exception:
        return None


def _runtime_candidates():
    """Fallback chain. XNNPACK is fastest, but it cannot prepare some INT8 graphs
    (e.g. MobileNetV3 blocks in some TensorFlow builds): the built-in kernels always work."""
    import warnings
    warnings.filterwarnings("ignore", message=".*tf.lite.Interpreter is deprecated.*")
    tf = _tf(); lit = _litert(); c = []
    if lit is not None:
        c.append(("litert-xnnpack", lit.Interpreter, {}))
    c.append(("tflite-xnnpack", tf.lite.Interpreter, {}))
    if lit is not None:
        c.append(("litert-builtin", lit.Interpreter,
                  {"experimental_op_resolver_type": lit.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES}))
    c.append(("tflite-builtin", tf.lite.Interpreter,
              {"experimental_op_resolver_type": tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES}))
    skip = [x for x in os.environ.get("TMN_SKIP_RUNTIMES", "").split(",") if x]
    return [x for x in c if x[0] not in skip]


_RUNTIME_OK = {}        # model fingerprint -> runtime that worked (skips known failures)


def make_interpreter(model_bytes, threads=2, batch=None, verbose=True):
    """Allocated, smoke-tested interpreter; tries each runtime in turn. Returns (it, name)."""
    global LAST_BACKEND
    key = (len(model_bytes), hash(model_bytes))
    cands = _runtime_candidates()
    if key in _RUNTIME_OK:                    # the runtime that worked last time goes first
        cands = sorted(cands, key=lambda c: c[0] != _RUNTIME_OK[key])
    err = None
    for name, cls, kw in cands:
        try:
            it = cls(model_content=model_bytes, num_threads=threads, **kw)
            ind = it.get_input_details()[0]
            if batch:
                it.resize_tensor_input(ind["index"], [batch] + list(ind["shape"][1:]))
            it.allocate_tensors()
            ind = it.get_input_details()[0]
            it.set_tensor(ind["index"], np.zeros(ind["shape"], ind["dtype"])); it.invoke()
            LAST_BACKEND = name; _RUNTIME_OK[key] = name
            return it, name
        except Exception as e:
            err = e
            if verbose:
                print("   runtime %s cannot run this model (%s) -> trying the next one"
                      % (name, str(e).split("\n")[0][:90]))
    raise err


def quantize_input(xb_u8, dtype, scale, zp):
    """Map uint8 camera pixels to the model's input tensor."""
    if dtype == np.uint8:
        if abs(scale - 1.0) < 1e-3 and zp == 0:
            return xb_u8
        return np.clip(np.round(xb_u8 / scale + zp), 0, 255).astype(np.uint8)
    if dtype == np.int8:                      # students: x = (p-128)/128
        if abs(scale - 1 / 128) < 1e-7 and zp == 0:
            return (xb_u8.astype(np.int16) - 128).astype(np.int8)
        return np.clip(np.round(to_unit(xb_u8) / scale + zp), -128, 127).astype(np.int8)
    return xb_u8.astype(np.float32)           # float models (teacher FP32/FP16/dynamic)


def tflite_predict(model_bytes, X_u8, bs=16, threads=2):
    """Batched LiteRT/TFLite inference; returns P(fire) as float32 (runtime in LAST_BACKEND)."""
    it, _ = make_interpreter(model_bytes, threads, batch=bs)
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    in_s, in_z = ind["quantization"]; out_s, out_z = outd["quantization"]
    out = np.empty(len(X_u8), np.float32)
    for s in range(0, len(X_u8), bs):
        xb = X_u8[s:s + bs]
        n = len(xb)
        if n < bs:
            xb = np.concatenate([xb, np.zeros((bs - n,) + xb.shape[1:], xb.dtype)])
        xin = quantize_input(xb, ind["dtype"], in_s, in_z)
        it.set_tensor(ind["index"], xin); it.invoke()
        o = it.get_tensor(outd["index"]).reshape(bs, -1)[:n, 0]
        out[s:s + n] = (o.astype(np.float32) - out_z) * out_s if outd["dtype"] != np.float32 else o
    return out


def tflite_latency(model_bytes, threads=1, warm=50, runs=200):
    """Median single-image latency (ms) with the first runtime that can run the model."""
    it, _ = make_interpreter(model_bytes, threads, batch=None)
    ind = it.get_input_details()[0]
    x = np.random.default_rng(0).integers(0, 255, ind["shape"]).astype(ind["dtype"])
    for _ in range(warm):
        it.set_tensor(ind["index"], x); it.invoke()
    ts = []
    for _ in range(runs):
        t = time.perf_counter(); it.set_tensor(ind["index"], x); it.invoke()
        ts.append(time.perf_counter() - t)
    return float(np.median(ts) * 1e3)


_TFLM_PROBE = r'''
import sys, numpy as np
from tflite_micro.python.tflite_micro import runtime as r
buf = open(sys.argv[1], "rb").read()
it = r.Interpreter.from_bytes(buf)
d = it.get_input_details(0)
it.set_input(np.zeros(d["shape"], d["dtype"]), 0); it.invoke(); it.print_allocations()
'''


def tflm_arena_bytes(tflite_path):
    """Exact tensor-arena size recorded by the TFLite Micro RecordingMicroAllocator."""
    probe = "/tmp/_tmn_tflm_probe.py"
    open(probe, "w").write(_TFLM_PROBE)
    r = subprocess.run([sys.executable, probe, tflite_path], capture_output=True, text=True)
    m = re.search(r"Arena allocation total (\d+)", r.stdout + r.stderr)
    return int(m.group(1)) if m else None


def tflm_predict_u8(tflite_path, X_u8):
    """Run the TFLite Micro reference runtime (the MCU arithmetic) on uint8 images."""
    from tflite_micro.python.tflite_micro import runtime as r
    it = r.Interpreter.from_bytes(open(tflite_path, "rb").read())
    d = it.get_input_details(0); q = d["quantization_parameters"]
    s, z = float(q["scales"][0]), int(q["zero_points"][0])
    out = np.empty(len(X_u8), np.int32)
    for i in range(len(X_u8)):
        it.set_input(quantize_input(X_u8[i:i + 1], d["dtype"], s, z), 0); it.invoke()
        out[i] = int(np.asarray(it.get_output(0)).ravel()[0])
    return out


def tflite_raw_u8(model_bytes, X_u8):
    it, _ = make_interpreter(model_bytes, 2, batch=None)
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    s, z = ind["quantization"]
    out = np.empty(len(X_u8), np.int32)
    for i in range(len(X_u8)):
        it.set_tensor(ind["index"], quantize_input(X_u8[i:i + 1], ind["dtype"], s, z)); it.invoke()
        out[i] = int(it.get_tensor(outd["index"]).ravel()[0])
    return out


def _vela_cmd():
    exe = shutil.which("vela") or os.path.join(os.path.dirname(sys.executable), "vela")
    return [exe] if os.path.exists(exe) else [sys.executable, "-m", "ethosu.vela"]


def vela_estimate(tflite_path, outdir, accel="ethos-u55-128"):
    outdir = os.path.join(outdir, accel)          # one folder per accelerator configuration
    shutil.rmtree(outdir, ignore_errors=True)
    os.makedirs(outdir, exist_ok=True)
    try:
        r = subprocess.run(_vela_cmd() + [tflite_path, "--accelerator-config", accel,
                                          "--output-dir", outdir], capture_output=True, text=True)
    except OSError as e:
        print("Vela could not be started (%s); the NPU estimate is skipped." % e)
        return None
    stem = os.path.splitext(os.path.basename(tflite_path))[0]
    cs = glob.glob(os.path.join(outdir, stem + "_summary_*.csv"))
    if not cs:
        print("Vela failed:", r.stdout[-800:], r.stderr[-800:])
        return None
    import csv
    row = list(csv.DictReader(open(cs[0])))[0]
    return dict(vela_accel=accel, vela_ms=float(row["inference_time"]) * 1e3,
                vela_sram_KiB=float(row["sram_memory_used"]),
                vela_flash_KiB=float(row.get("off_chip_flash_memory_used", 0) or 0),
                vela_macs=int(float(row["nn_macs"])), vela_cycles=int(float(row["cycles_total"])))


def count_macs(model, res):
    """Multiply-accumulate count of the float graph (TF profiler FLOPs / 2)."""
    tf = _tf()
    from tensorflow.python.framework.convert_to_constants import \
        convert_variables_to_constants_v2_as_graph
    f = tf.function(lambda x: model(x, training=False)).get_concrete_function(
        tf.TensorSpec([1, res, res, 3], tf.float32))
    frozen, _ = convert_variables_to_constants_v2_as_graph(f)
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    opts["output"] = "none"
    prof = tf.compat.v1.profiler.profile(graph=frozen.graph, run_meta=tf.compat.v1.RunMetadata(),
                                         cmd="op", options=opts)
    return int(prof.total_float_ops // 2)


# ---------------------------------------------------------------------------
# 9. Weight-only fake quantisation (bit-width sweep)
# ---------------------------------------------------------------------------
def _iter_layers(model):
    keras = keras_mod()
    for l in model.layers:
        if isinstance(l, keras.Model):
            yield from _iter_layers(l)
        else:
            yield l


def fake_quant_weights(model, bits):
    """Per-output-channel symmetric b-bit quantisation of every Conv/Depthwise/Dense
    kernel, in place. Returns a backup list for restore_weights()."""
    keras = keras_mod()
    qmax = 2 ** (bits - 1) - 1
    backup = []
    for l in _iter_layers(model):
        if isinstance(l, keras.layers.DepthwiseConv2D):
            axes = (0, 1)                      # kernel (kh, kw, channels, multiplier)
        elif isinstance(l, (keras.layers.Conv2D, keras.layers.Dense)):
            axes = tuple(range(len(l.kernel.shape) - 1))   # per output channel
        else:
            continue
        var = l.depthwise_kernel if isinstance(l, keras.layers.DepthwiseConv2D) else l.kernel
        W = var.numpy()
        backup.append((var, W.copy()))
        s = np.maximum(np.abs(W).max(axis=axes, keepdims=True), 1e-12) / qmax
        var.assign(np.clip(np.round(W / s), -qmax, qmax) * s)
    return backup


def restore_weights(backup):
    for var, W in backup:
        var.assign(W)


# ---------------------------------------------------------------------------
# 10. Metrics and statistics
# ---------------------------------------------------------------------------
def ece_score(y, p, bins=15):
    """Expected calibration error of P(fire) (binary, equal-width bins)."""
    y = np.asarray(y); p = np.asarray(p, float)
    e = np.linspace(0, 1, bins + 1); tot = 0.0
    for a, b in zip(e[:-1], e[1:]):
        m = (p > a) & (p <= b) if a > 0 else (p >= a) & (p <= b)
        if m.any():
            tot += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(tot)


def metrics(y, p, thr=0.5):
    from sklearn.metrics import roc_auc_score, average_precision_score
    y = np.asarray(y).astype(int); p = np.asarray(p, float); yp = (p > thr).astype(int)
    tp = int(((yp == 1) & (y == 1)).sum()); tn = int(((yp == 0) & (y == 0)).sum())
    fp = int(((yp == 1) & (y == 0)).sum()); fn = int(((yp == 0) & (y == 1)).sum())
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); prec = tp / max(tp + fp, 1)
    two = len(np.unique(y)) == 2
    return dict(n=len(y), acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, recall=rec,
                specificity=spec, precision=prec,
                f1=2 * prec * rec / max(prec + rec, 1e-12),
                auc=roc_auc_score(y, p) if two else np.nan,
                ap=average_precision_score(y, p) if two else np.nan,
                ece=ece_score(y, p), brier=float(np.mean((p - y) ** 2)),
                tp=tp, tn=tn, fp=fp, fn=fn)


def wilson(k, n, z=1.959964):
    if n == 0:
        return (np.nan, np.nan)
    ph = k / n; d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


def mcnemar_p(y, pa, pb, thr=0.5):
    """Two-sided McNemar test on paired correctness (exact if discordant < 25)."""
    from statsmodels.stats.contingency_tables import mcnemar
    a = ((np.asarray(pa) > thr).astype(int) == y); b = ((np.asarray(pb) > thr).astype(int) == y)
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    if n01 + n10 == 0:
        return 1.0, n01, n10
    r = mcnemar([[int((a & b).sum()), n01], [n10, int((~a & ~b).sum())]],
                exact=(n01 + n10) < 25, correction=True)
    return float(r.pvalue), n01, n10


def holm(pvals):
    from statsmodels.stats.multitest import multipletests
    return multipletests(pvals, method="holm")[1]


def flips(y, p_ref, p_new, thr=0.5):
    a = (np.asarray(p_ref) > thr).astype(int); b = (np.asarray(p_new) > thr).astype(int)
    f = a != b
    return dict(flip_rate=float(f.mean()), n_flip=int(f.sum()),
                new_missed_fires=int((f & (y == 1) & (b == 0)).sum()),
                new_false_alarms=int((f & (y == 0) & (b == 1)).sum()),
                fixed=int((f & (b == y)).sum()), broken=int((f & (a == y)).sum()))


def paired_bootstrap(y, pa, pb, fn="bacc", B=2000, seed=0, thr=0.5):
    """95% CI of metric(pb) - metric(pa) by paired bootstrap over images."""
    rng = np.random.default_rng(seed); y = np.asarray(y); n = len(y)
    f = (lambda yy, pp: metrics(yy, pp, thr)[fn])
    d0 = f(y, pb) - f(y, pa); ds = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        if len(np.unique(y[i])) < 2:
            continue
        ds.append(f(y[i], pb[i]) - f(y[i], pa[i]))
    lo, hi = np.percentile(ds, [2.5, 97.5])
    return float(d0), float(lo), float(hi)


# ---------------------------------------------------------------------------
# 11. Result bookkeeping (resumable)
# ---------------------------------------------------------------------------
def labels(P):
    return dict(np.load(os.path.join(P["data"], "labels.npz"), allow_pickle=True))


def teacher_soft(P):
    """Teacher-ensemble soft labels (logits) for every pool block: {'orig_0': array, ...}."""
    d = np.load(os.path.join(P["data"], "teacher_soft.npz"))
    return {k: d[k] for k in d.files}


def pred_path(P, model_id):
    return os.path.join(P["preds"], model_id + ".npz")


def have_preds(P, model_id):
    return os.path.exists(pred_path(P, model_id))


def save_preds(P, model_id, preds, meta, idx=None):
    """Per-image P(fire). If idx is given, preds[ds] belongs to images idx[ds] of set ds."""
    extra = {"idx_" + k: np.asarray(v, np.int32) for k, v in (idx or {}).items()}
    np.savez_compressed(pred_path(P, model_id), meta=json.dumps(meta), **extra,
                        **{k: np.asarray(v, np.float32) for k, v in preds.items()})


def load_preds(P, model_id):
    """Returns (preds, meta, idx); idx[ds] is None when the full set was evaluated."""
    d = np.load(pred_path(P, model_id), allow_pickle=True)
    preds = {k: d[k] for k in d.files if k != "meta" and not k.startswith("idx_")}
    idx = {k: (d["idx_" + k] if "idx_" + k in d.files else None) for k in preds}
    return preds, json.loads(str(d["meta"])), idx


def metric_rows(P, model_id, preds, meta, idx=None):
    lab = labels(P); rows = []
    for ds in EVALSETS:
        if ds in preds:
            y = lab["y_" + ds] if not idx or idx.get(ds) is None else lab["y_" + ds][idx[ds]]
            r = dict(meta); r.update(model_id=model_id, dataset=ds)
            r.update(metrics(y, preds[ds])); rows.append(r)
    return rows


def ladder_subsets(P, n=1000, seed=42):
    """Fixed evaluation subsets for the teacher's TFLite ladder: the whole in-domain test set
    and a label-stratified random sample of n images from each shifted set.
    A per-class random order is drawn once and saved (data/ladder_order.json); the subset of
    size n is the first round(n * class share) images of each class, so subsets for different
    n are nested (a 600-image subset lies inside the 1,000-image one)."""
    f = os.path.join(P["data"], "ladder_order.json")
    lab = labels(P)
    if os.path.exists(f):
        order = {k: np.array(v, np.int64) for k, v in load_json(f).items()}
    else:
        rng = np.random.default_rng(seed); order = {}
        for ds in EVALSETS:
            y = lab["y_" + ds]
            for c in (0, 1):
                order[f"{ds}_{c}"] = rng.permutation(np.where(y == c)[0])
        save_json({k: v.tolist() for k, v in order.items()}, f)
    out = {}
    for ds in EVALSETS:
        y = lab["y_" + ds]
        if ds == "din" or len(y) <= n:
            out[ds] = np.arange(len(y)); continue
        pick = [order[f"{ds}_{c}"][:int(round(n * np.sum(y == c) / len(y)))] for c in (0, 1)]
        out[ds] = np.sort(np.concatenate(pick))
    return out


def common_index(idx_list, n_total):
    """Images shared by several prediction sets (None = the full set)."""
    keep = np.ones(n_total, bool)
    for ix in idx_list:
        if ix is not None:
            m = np.zeros(n_total, bool); m[np.asarray(ix)] = True; keep &= m
    return np.where(keep)[0]


def preds_on(pr_ds, ix_ds, target):
    """Predictions of one model restricted to the image indices in `target`."""
    if ix_ds is None:
        return np.asarray(pr_ds)[target]
    pos = {int(i): k for k, i in enumerate(np.asarray(ix_ds))}
    return np.asarray(pr_ds)[[pos[int(i)] for i in target]]


def append_rows(path, rows):
    import pandas as pd
    df = pd.DataFrame(rows)
    if os.path.exists(path):
        old = pd.read_csv(path)
        old = old[~old["model_id"].isin(df["model_id"].unique())]
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(path, index=False)
    return df


def load_metrics(P):
    """All metric rows from every notebook (each notebook writes its own CSV)."""
    import pandas as pd
    fs = sorted(glob.glob(os.path.join(P["results"], "metrics_nb*.csv")))
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()


def load_eval_arrays(P, res=None):
    """Evaluation images at 224 (or resized to res), keyed by EVALSETS."""
    d = np.load(os.path.join(P["data"], "eval_224.npz"))
    X = {k: d["X_" + k] for k in EVALSETS}
    if res and res != CFG["IMG"]:
        X = {k: resize_u8(v, res) for k, v in X.items()}
    return X


def env_report():
    rep = dict(python=sys.version.split()[0])
    for pkg in ["tensorflow", "tf_keras", "tensorflow_model_optimization", "numpy",
                "tflite_micro", "ethos-u-vela", "kagglehub", "imagehash"]:
        try:
            import importlib.metadata as md
            rep[pkg] = md.version(pkg.replace("_", "-") if pkg != "tflite_micro" else "tflite-micro")
        except Exception:
            try:
                rep[pkg] = md.version(pkg)
            except Exception:
                rep[pkg] = None
    try:
        tf = _tf()
        rep["gpu"] = [d.name for d in tf.config.list_physical_devices("GPU")]
    except Exception:
        pass
    return rep

Writing /content/tmn_common.py


In [ ]:
#@title 3 · Mounting and Loading
import os, sys, importlib
os.environ["TF_USE_LEGACY_KERAS"] = "1"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive not mounted (not on Colab?):", e)
sys.path.insert(0, "/content")
import tmn_common as C; importlib.reload(C)
P = C.setup_paths()
import numpy as np, pandas as pd, tensorflow as tf
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass
import tf_keras as keras
print("Notebook version:", C.VERSION)
ENV = C.env_report(); print(ENV)
C.save_json(ENV, os.path.join(P["logs"], "env_nb2.json"))
print("All outputs are saved under:", P["root"])
if False and not tf.config.list_physical_devices("GPU") and not C.CFG["TEST_MODE"]:
    raise SystemExit("No GPU. In Colab: Runtime > Change runtime type > T4 GPU, then re-run.")

Mounted at /content/drive
Notebook version: 29 Sep 2026
{'python': '3.13.15', 'tensorflow': '2.20.0', 'tf_keras': '2.20.1', 'tensorflow_model_optimization': None, 'numpy': '2.1.3', 'tflite_micro': '0.dev20251201172741', 'ethos-u-vela': '5.2.0', 'kagglehub': '1.0.2', 'imagehash': None, 'gpu': []}
All outputs are saved under: /content/drive/MyDrive/TinyMWAMNet


In [ ]:
#@title 4 · Load data, teacher backbones, calibration sets and the fixed evaluation subsets
import time
LADDER_N = 1000                      # images per shifted set (lower it, e.g. 600, if your runtime is slow)
if C.CFG["TEST_MODE"]:
    LADDER_N = 16                    # pipeline test only
lab = C.labels(P); X = C.load_eval_arrays(P)
din = np.load(os.path.join(P["data"], "din_224.npz")); X_tr = din["X_tr"]
med = C.load_json(os.path.join(P["data"], "teacher_median_seed.json"))["median_seed"]
d_net, m_net = C.teacher_backbones()
rng = np.random.default_rng(C.CFG["SEED"]); n = C.CFG["CALIB_N"]
def calib(blocks, n):
    # n (block, index) pairs sampled without replacement, regenerated deterministically
    pairs = [(b, i) for b in blocks for i in range(len(X_tr))]
    pick = rng.choice(len(pairs), n, replace=False); out = []
    for kind, c in blocks:
        idx = sorted(pairs[j][1] for j in pick if pairs[j][0] == (kind, c))
        if idx:
            out.append(C.make_block(X_tr, kind, c, idx))
    return np.concatenate(out)
CAL = {"din": calib(C.teacher_blocks(), n),
       "conf": calib([("conf", k) for k in range(1, C.CFG["CONF_COPIES"] + 1)], n)}
SUB = C.ladder_subsets(P, n=LADDER_N)
XS = {ds: X[ds][SUB[ds]] for ds in C.EVALSETS}
print({k: v.shape for k, v in CAL.items()}, "| median seed:", med)
print("Evaluation images per model:", {ds: len(v) for ds, v in XS.items()})

12683000/12683000 [==============================] - 0s 0us/step
{'din': (256, 224, 224, 3), 'conf': (256, 224, 224, 3)} | median seed: 46
Evaluation images per model: {'din': 380, 'wild': 1000, 'dfire': 1000}


In [ ]:
#@title 5 · Convert + evaluate the teacher ladder (resumable; prints speed and remaining time)
LADDER_SEEDS = [med] + [s for s in C.CFG["SEEDS"] if s != med][:2]      # 3 seeds incl. the median
PLAN = [("fp16", None, [med]), ("dyn", None, [med]),
        ("int8", "din", LADDER_SEEDS), ("int8c", "conf", LADDER_SEEDS)]
jobs = sorted([(s, var, cal) for var, cal, seeds in PLAN for s in seeds])
todo = [j for j in jobs if not C.have_preds(P, f"T_{j[1]}_s{j[0]}")]
print("Models:", len(jobs), "| already done:", len(jobs) - len(todo), "| to run:", [f"T_{v}_s{s}" for s, v, c in todo])
_T = {}
def teacher(s):
    if s not in _T:
        _T.clear(); keras.backend.clear_session()
        _T[s] = C.build_teacher_e2e(os.path.join(P["models"], f"teacher_head_s{s}.weights.h5"), d_net, m_net)
    return _T[s]
n_img = sum(len(v) for v in XS.values()); spent = []
for k, (s, var, cal) in enumerate(todo):
    mid = f"T_{var}_s{s}"; f = os.path.join(P["tflite"], mid + ".tflite")
    t0 = time.time()
    if os.path.exists(f):
        b = open(f, "rb").read()                     # converted in an earlier session
    else:
        mode = "int8" if var.startswith("int8") else var
        b = C.tflite_convert(teacher(s), 224, mode, C.rep_from(CAL[cal]) if cal else None)
        open(f, "wb").write(b)
    tc = time.time() - t0
    pr = {ds: C.tflite_predict(b, XS[ds], bs=16, threads=2) for ds in C.EVALSETS}
    te = time.time() - t0 - tc
    meta = dict(family="teacher", arch="MWAMNet", res=224, regime="-", quant=var, seed=s,
                calib=cal or "-", size_bytes=len(b), runtime=C.LAST_BACKEND, subset="ladder%d" % LADDER_N)
    C.save_preds(P, mid, pr, meta, idx=SUB); rows = C.metric_rows(P, mid, pr, meta, idx=SUB)
    C.append_rows(os.path.join(P["results"], "metrics_nb2.csv"), rows)
    spent.append(time.time() - t0); left = len(todo) - k - 1
    print("%-12s %.1f MB | convert %.0fs | eval %.1f min (%.1f img/s, %s) | acc din/wild/dfire = %s | ~%.0f min left"
          % (mid, len(b) / 2**20, tc, te / 60, n_img / max(te, 1e-9), C.LAST_BACKEND,
             " / ".join("%.4f" % r["acc"] for r in rows), left * np.mean(spent) / 60))
print("Teacher ladder complete.")

Models: 8 | already done: 0 | to run: ['T_int8_s42', 'T_int8c_s42', 'T_int8_s43', 'T_int8c_s43', 'T_dyn_s46', 'T_fp16_s46', 'T_int8_s46', 'T_int8c_s46']
   runtime litert-xnnpack cannot run this model (failed to create XNNPACK runtimeNode number 643 (TfLiteXNNPackDelegate) failed to prepare.) -> trying the next one
   runtime tflite-xnnpack cannot run this model (failed to create XNNPACK runtimeNode number 643 (TfLiteXNNPackDelegate) failed to prepare.) -> trying the next one
T_int8_s42   24.5 MB | convert 2s | eval 19.0 min (2.1 img/s, litert-builtin) | acc din/wild/dfire = 0.8921 / 0.6040 / 0.7280 | ~133 min left
   runtime litert-xnnpack cannot run this model (failed to create XNNPACK runtimeNode number 643 (TfLiteXNNPackDelegate) failed to prepare.) -> trying the next one
   runtime tflite-xnnpack cannot run this model (failed to create XNNPACK runtimeNode number 643 (TfLiteXNNPackDelegate) failed to prepare.) -> trying the next one
T_int8c_s42  24.5 MB | convert 534s | eval 18.7 m

In [ ]:
#@title 6 · Latency (batch 1), TFLite-Micro arena and Vela estimate for the deployed teacher
T = C.build_teacher_e2e(os.path.join(P["models"], f"teacher_head_s{med}.weights.h5"), d_net, m_net)
lat = []
for var in ["fp32", "fp16", "dyn", "int8"]:
    f = os.path.join(P["tflite"], f"T_{var}_s{med}.tflite")
    b = open(f, "rb").read() if os.path.exists(f) else C.tflite_convert(
        T, 224, var, C.rep_from(CAL["din"]) if var == "int8" else None)
    ms = C.tflite_latency(b, threads=1, warm=3, runs=20)    # single thread; few repetitions (big model)
    lat.append(dict(model="MWAMNet", quant=var, threads=1, size_MiB=len(b) / 2**20,
                    latency_ms=ms, runtime=C.LAST_BACKEND))
    print(lat[-1])
pd.DataFrame(lat).to_csv(os.path.join(P["results"], "latency_teacher.csv"), index=False)
f8 = os.path.join(P["tflite"], f"T_int8_s{med}.tflite")
dep = dict(model="MWAMNet", params=int(T.count_params()), flash_bytes=os.path.getsize(f8),
           tflm_arena_bytes=C.tflm_arena_bytes(f8), macs=C.count_macs(T, 224))
RUN_VELA_TEACHER = True
if RUN_VELA_TEACHER:
    v = C.vela_estimate(f8, os.path.join(P["tmp"], "vela_teacher"))
    dep.update(v or {})
C.save_json(dep, os.path.join(P["results"], "deploy_teacher.json")); print(dep)

{'model': 'MWAMNet', 'quant': 'fp32', 'threads': 1, 'size_MiB': 93.73154067993164, 'latency_ms': 220.1019494996217, 'runtime': 'litert-xnnpack'}
{'model': 'MWAMNet', 'quant': 'fp16', 'threads': 1, 'size_MiB': 47.02848434448242, 'latency_ms': 187.10627799919166, 'runtime': 'litert-xnnpack'}
{'model': 'MWAMNet', 'quant': 'dyn', 'threads': 1, 'size_MiB': 24.686302185058594, 'latency_ms': 342.1775284987234, 'runtime': 'litert-xnnpack'}
{'model': 'MWAMNet', 'quant': 'int8', 'threads': 1, 'size_MiB': 24.53485107421875, 'latency_ms': 456.9026089993713, 'runtime': 'litert-builtin'}


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


{'model': 'MWAMNet', 'params': 24811457, 'flash_bytes': 25726656, 'tflm_arena_bytes': 3126128, 'macs': 4511914038, 'vela_accel': 'ethos-u55-128', 'vela_ms': 475.562342, 'vela_sram_KiB': 2712.046875, 'vela_flash_KiB': 21157.46875, 'vela_macs': 4527738529, 'vela_cycles': 237781171}


In [ ]:
#@title 7 · Summary of the teacher ladder on the same images (mean over seeds)
ladder = {}                                            # FP32 (Keras, notebook 01) + TFLite ladder
for f in sorted(os.listdir(P["preds"])):
    if f.startswith("T_") and f.endswith(".npz") and not f.startswith("T_w") and "ens" not in f:
        pr, meta, ix = C.load_preds(P, f[:-4])
        if meta["seed"] in LADDER_SEEDS:
            ladder[f[:-4]] = (pr, meta, ix)
common = {ds: C.common_index([ix.get(ds) for _, _, ix in ladder.values()], len(lab["y_" + ds]))
          for ds in C.EVALSETS}                        # images evaluated by every ladder model
rows, fl = [], []
ref = {m["seed"]: (pr, ix) for pr, m, ix in ladder.values() if m["quant"] == "fp32"}
for mid, (pr, meta, ix) in ladder.items():
    for ds in C.EVALSETS:
        y = lab["y_" + ds][common[ds]]; p = C.preds_on(pr[ds], ix.get(ds), common[ds])
        rows.append(dict(quant=meta["quant"], seed=meta["seed"], dataset=ds, **C.metrics(y, p)))
        if meta["quant"] != "fp32" and meta["seed"] in ref:
            p0 = C.preds_on(ref[meta["seed"]][0][ds], ref[meta["seed"]][1].get(ds), common[ds])
            fl.append(dict(quant=meta["quant"], dataset=ds, **C.flips(y, p0, p)))
print("Images per dataset:", {ds: len(v) for ds, v in common.items()}, "| seeds:", LADDER_SEEDS)
tab = pd.DataFrame(rows).groupby(["quant", "dataset"])[["acc", "bacc", "recall", "auc", "ece"]].agg(["mean", "count"])
print(tab.round(4).to_string())
if fl:
    print("\nPrediction flips versus FP32 of the same seed (mean over seeds):")
    print(pd.DataFrame(fl).groupby(["quant", "dataset"])[["flip_rate", "new_missed_fires", "new_false_alarms"]]
          .mean().round(4).to_string())


Images per dataset: {'din': 380, 'wild': 1000, 'dfire': 1000} | seeds: [46, 42, 43]
                  acc          bacc        recall           auc           ece      
                 mean count    mean count    mean count    mean count    mean count
quant dataset                                                                      
dyn   dfire    0.8105     2  0.7994     2  0.7605     2  0.8961     2  0.1333     2
      din      0.9908     2  0.9908     2  0.9895     2  0.9997     2  0.0104     2
      wild     0.7295     2  0.6789     2  0.4548     2  0.7510     2  0.2215     2
fp16  dfire    0.8210     1  0.8160     1  0.7983     1  0.9099     1  0.0968     1
      din      0.9895     1  0.9895     1  0.9895     1  0.9999     1  0.0124     1
      wild     0.7390     1  0.6890     1  0.4677     1  0.7650     1  0.1935     1
fp32  dfire    0.8107     3  0.8162     3  0.8357     3  0.9027     3  0.1388     3
      din      0.9895     3  0.9895     3  0.9895     3  0.9998     3  0.014